# 02 — CBraMod PhysioNet-MI: reproduce with the authors' code

Run your existing **`00_colab_setup(2).ipynb`** first, then this notebook in the **same Colab runtime**. Do not change your GitHub setup notebook.

- Uses the **official CBraMod** preprocessing, model, fine-tuning, and evaluation scripts (pinned to one commit).
- Uses the authors' original **LMDB** dataset format, 70/19/20 subject split, training defaults, and pretrained checkpoint.
- Extracts and **saves EDF data and processed LMDB in Drive once**; subsequent Colab sessions copy the already-processed LMDB to local disk instead of preprocessing again.
- Runs fine-tuning from `/content` for speed and saves the final checkpoint and log to Drive.

Only technical adjustments: link the authors' fixed input/output paths to the Colab directories and fix repeated LMDB openings in their loader. **No changes to EEG transformations, model, loss, optimizer, epochs, or evaluation.**

Official code: https://github.com/wjq-learning/CBraMod

## 1 · Paths matching your setup notebook

In [1]:
from pathlib import Path
import os, sys, subprocess, shutil, re, pickle, csv
from zipfile import ZipFile
from pathlib import PurePosixPath

# Same paths already used by your existing 00 setup notebook.
REPO_DIR = Path('/content/cbramod-head-interp')   # your GitHub project checkout
DATA_DIR = Path('/content/data')                  # temporary Colab disk
DRIVE_ROOT = Path('/content/drive/MyDrive/CBraMod-MI')
ZIP_PATH = DRIVE_ROOT / 'eeg-motor-movementimagery-dataset-1.0.0.zip'

# Persistent, organized files (Google Drive).
DRIVE_DATA = DRIVE_ROOT / 'data' / 'physionet_mi'
DRIVE_EDF = DRIVE_DATA / 'files'
DRIVE_LMDB = DRIVE_DATA / 'processed_average'
DRIVE_WEIGHTS = DRIVE_ROOT / 'models' / 'pretrained' / 'cbramod' / 'pretrained_weights.pth'
DRIVE_RESULTS = DRIVE_ROOT / 'results' / 'physionet_mi_4class'

# Training always reads from local Colab disk, not directly from Google Drive.
LOCAL_WORK = DATA_DIR / 'physionet_mi'
LOCAL_EDF = LOCAL_WORK / 'files'
LOCAL_LMDB = LOCAL_WORK / 'processed_average'
LOCAL_RUNS = LOCAL_WORK / 'runs'
AUTHORS_REPO = Path('/content/CBraMod')           # separate official PUBLIC repository

DRIVE_DATA.mkdir(parents=True, exist_ok=True)
LOCAL_WORK.mkdir(parents=True, exist_ok=True)
print('Your GitHub project:', REPO_DIR)
print('Drive data cache:', DRIVE_DATA)
print('Local training files:', LOCAL_WORK)

Your GitHub project: /content/cbramod-head-interp
Drive data cache: /content/drive/MyDrive/CBraMod-MI/data/physionet_mi
Local training files: /content/data/physionet_mi


## 2 · Get the official CBraMod code and dependencies

Your personal GitHub repo is **not modified**. This is a separate checkout of the authors' public implementation.

In [2]:
OFFICIAL_COMMIT = 'b9e961003214326972c567eff390e75b0287e32a'
if not (AUTHORS_REPO / '.git').exists():
    subprocess.run(['git', 'clone', 'https://github.com/wjq-learning/CBraMod.git',
                    str(AUTHORS_REPO)], check=True)
subprocess.run(['git', '-C', str(AUTHORS_REPO), 'checkout', '--quiet', OFFICIAL_COMMIT], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                '-r', str(AUTHORS_REPO / 'requirements.txt'), 'huggingface_hub'], check=True)

import lmdb, torch
from huggingface_hub import hf_hub_download
print("Authors' commit:", OFFICIAL_COMMIT)
print('GPU:', torch.cuda.get_device_name(0))

Authors' commit: b9e961003214326972c567eff390e75b0287e32a
GPU: Tesla T4


## 3 · Keep the extracted EDF recordings in Drive (first run only)

The authors use EDF files named `files/S001/S001R04.edf` etc. We extract the **six imagery runs** (04, 06, 08, 10, 12, 14) for 109 subjects. The ZIP stays where your setup notebook expects it. If the LMDB cache already exists, this cell skips the EDF copy entirely.

In [3]:
EDF_DONE = DRIVE_DATA / 'edf_complete.txt'
LMDB_DONE = DRIVE_DATA / 'lmdb_complete.txt'

if LMDB_DONE.is_file() and (DRIVE_LMDB / 'data.mdb').is_file():
    print('Processed LMDB already saved in Drive; EDF extraction is unnecessary.')
elif EDF_DONE.is_file():
    print('Copying saved EDF recordings to Colab temporary disk...')
    shutil.copytree(DRIVE_EDF, LOCAL_EDF, dirs_exist_ok=True)
else:
    print('Extracting six imagery runs from ZIP (one-time operation)...')
    with ZipFile(ZIP_PATH) as archive:
        edfs = []
        for info in archive.infolist():
            name = PurePosixPath(info.filename).name
            if re.fullmatch(r'S\d{3}R(04|06|08|10|12|14)\.edf', name, flags=re.I):
                filename = name[:7].upper() + '.edf'
                edfs.append((info, filename))
        assert len(edfs) == 654, f'Expected 654 imagery EDF recordings, found {len(edfs)}'
        for info, filename in edfs:
            dst = LOCAL_EDF / filename[:4] / filename
            dst.parent.mkdir(parents=True, exist_ok=True)
            if dst.is_file() and dst.stat().st_size == info.file_size:
                continue
            with archive.open(info) as source, dst.open('wb') as output:
                shutil.copyfileobj(source, output)

    print('Saving extracted EDF folder to Google Drive...')
    shutil.copytree(LOCAL_EDF, DRIVE_EDF, dirs_exist_ok=True)
    EDF_DONE.write_text('654 imagery EDF recordings extracted from original PhysioNet-MI ZIP\n')
    print('Saved:', DRIVE_EDF)

Extracting six imagery runs from ZIP (one-time operation)...
Saving extracted EDF folder to Google Drive...
Saved: /content/drive/MyDrive/CBraMod-MI/data/physionet_mi/files


## 4 · Run **the authors' unmodified preprocessing script** (first run only)

Their script has fixed Linux paths under `/data/datasets/...`. We use **directory links**, so the script itself is not edited. It still applies the authors' referencing, filtering, epoching, labels, and subject split. After preprocessing we save their LMDB to Drive. Later sessions simply copy it back to `/content`.

In [4]:
if LMDB_DONE.is_file() and (DRIVE_LMDB / 'data.mdb').is_file():
    print('Loading previously preprocessed LMDB from Drive (no reprocessing)...')
    shutil.copytree(DRIVE_LMDB, LOCAL_LMDB, dirs_exist_ok=True)
else:
    LOCAL_LMDB.mkdir(parents=True, exist_ok=True)
    authors_dataset_dir = Path('/data/datasets/eeg-motor-movementimagery-dataset-1.0.0')
    authors_dataset_dir.mkdir(parents=True, exist_ok=True)
    for name, target in [('files', LOCAL_EDF), ('processed_average', LOCAL_LMDB)]:
        link = authors_dataset_dir / name
        if link.is_symlink():
            link.unlink()
        elif link.exists():
            raise RuntimeError(f'Cannot replace existing directory: {link}')
        link.symlink_to(target, target_is_directory=True)

    # This runs the ORIGINAL preprocessing_physio.py directly (no path patch).
    preprocessing_log = LOCAL_WORK / 'preprocessing.log'
    print("Running authors' preprocessing script; log:", preprocessing_log, flush=True)
    with preprocessing_log.open('w') as log:
        process = subprocess.run([sys.executable, '-u', 'preprocessing/preprocessing_physio.py'],
                                 cwd=str(AUTHORS_REPO), stdout=log, stderr=subprocess.STDOUT)
    if process.returncode != 0:
        raise RuntimeError("Authors' preprocessing failed:\n" + preprocessing_log.read_text()[-4000:])

    print('Saving processed LMDB to Drive for future sessions...')
    shutil.copytree(LOCAL_LMDB, DRIVE_LMDB, dirs_exist_ok=True)
    LMDB_DONE.write_text(f'Preprocessed by official CBraMod commit {OFFICIAL_COMMIT}\n')
    print('Saved:', DRIVE_LMDB)

# Confirm the authors' train / validation / test LMDB keys exist.
db = lmdb.open(str(LOCAL_LMDB), readonly=True, lock=False, readahead=False)
with db.begin() as tx:
    keys = pickle.loads(tx.get(b'__keys__'))
    first = pickle.loads(tx.get(keys['train'][0].encode()))
db.close()
print('Train / val / test:', *(len(keys[k]) for k in ('train', 'val', 'test')))
print('Total trials:', sum(len(v) for v in keys.values()))
print('Example EEG shape:', first['sample'].shape)

Running authors' preprocessing script; log: /content/data/physionet_mi/preprocessing.log
Saving processed LMDB to Drive for future sessions...
Saved: /content/drive/MyDrive/CBraMod-MI/data/physionet_mi/processed_average
Train / val / test: 6300 1734 1803
Total trials: 9837
Example EEG shape: (64, 4, 200)


## 5 · Download the **authors'** pretrained weights (one time)

The checkpoint is saved to Drive for reuse; `finetune_main.py` loads the same checkpoint on each run.

In [5]:
if not DRIVE_WEIGHTS.is_file():
    downloaded = hf_hub_download(repo_id='weighting666/CBraMod',
                                 filename='pretrained_weights.pth')
    DRIVE_WEIGHTS.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(downloaded, DRIVE_WEIGHTS)
print('Pretrained checkpoint:', DRIVE_WEIGHTS)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


pretrained_weights.pth: reconstructing file:   0%|          |  0.00B / 19.8MB            

pretrained_weights.pth: downloading bytes:           |  0.00B            

Pretrained checkpoint: /content/drive/MyDrive/CBraMod-MI/models/pretrained/cbramod/pretrained_weights.pth


## 6 · Small LMDB loader compatibility fix

The authors' dataset loader opens the **same LMDB three times** for train/validation/test. On Colab's LMDB versions this can trigger an environment-already-open error. The change below **shares one read-only connection** and does not alter any labels, samples, model computations, or hyperparameters.

In [6]:
loader_path = AUTHORS_REPO / 'datasets' / 'physio_dataset.py'
source = loader_path.read_text()
old = 'self.db = lmdb.open(data_dir, readonly=True, lock=False, readahead=True, meminit=False)'
new = ("if not hasattr(CustomDataset, '_shared_db'):\n"
       "            CustomDataset._shared_db = lmdb.open(data_dir, readonly=True, lock=False, readahead=True, meminit=False)\n"
       "        self.db = CustomDataset._shared_db")
if old in source:
    source = source.replace(old, new, 1)
    compile(source, str(loader_path), 'exec')
    loader_path.write_text(source)
elif new not in source:
    raise RuntimeError("The authors' dataset loader has changed; review before patching.")
print('Original loader retained; only the shared read-only LMDB connection is adjusted.')

Original loader retained; only the shared read-only LMDB connection is adjusted.


## 7 · Fine-tune with **the original `finetune_main.py`**

The authors' script provides the training loop, AdamW, two learning-rate groups, cosine scheduler, label smoothing, gradient clipping, validation checkpoint selection, and held-out test evaluation. We keep its default **50 epochs, batch size 64, seed 3407** and specify only the dataset/model/checkpoint locations plus CUDA device `0` for Colab.

This is a **single-run reproduction**. The paper reports mean ± SD across five runs; the authors did not publish a documented list of all five seeds in the scripts examined.

In [7]:
SEED = 3407
LOCAL_RESULT = LOCAL_RUNS / f'seed_{SEED}'
LOCAL_RESULT.mkdir(parents=True, exist_ok=True)
command = [
    sys.executable, '-u', 'finetune_main.py',
    '--downstream_dataset', 'PhysioNet-MI',
    '--datasets_dir', str(LOCAL_LMDB),
    '--num_of_classes', '4',
    '--foundation_dir', str(DRIVE_WEIGHTS),
    '--model_dir', str(LOCAL_RESULT),
    '--cuda', '0',
    '--seed', str(SEED),
]
print('Running official fine-tuning:', ' '.join(command), flush=True)
log_path = LOCAL_RESULT / 'train.log'
with log_path.open('w') as log:
    process = subprocess.Popen(command, cwd=str(AUTHORS_REPO),
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='')
        log.write(line)
    status = process.wait()
if status != 0:
    raise RuntimeError(f'Fine-tuning exited with code {status}; see {log_path}')

Running official fine-tuning: /usr/bin/python3 -u finetune_main.py --downstream_dataset PhysioNet-MI --datasets_dir /content/data/physionet_mi/processed_average --num_of_classes 4 --foundation_dir /content/drive/MyDrive/CBraMod-MI/models/pretrained/cbramod/pretrained_weights.pth --model_dir /content/data/physionet_mi/runs/seed_3407 --cuda 0 --seed 3407
Namespace(seed=3407, cuda=0, epochs=50, batch_size=64, lr=0.0001, weight_decay=0.05, optimizer='AdamW', clip_value=1, dropout=0.1, classifier='all_patch_reps', downstream_dataset='PhysioNet-MI', datasets_dir='/content/data/physionet_mi/processed_average', num_of_classes=4, model_dir='/content/data/physionet_mi/runs/seed_3407', num_workers=16, label_smoothing=0.1, multi_lr=True, frozen=False, use_pretrained_weights=True, foundation_dir='/content/drive/MyDrive/CBraMod-MI/models/pretrained/cbramod/pretrained_weights.pth')
The downstream dataset is PhysioNet-MI
6300 1734 1803
9837
Model(
  (backbone): CBraMod(
    (patch_embedding): PatchEmb

## 8 · Save the authors' checkpoint, training log, and test results to Drive

In [8]:
DEST = DRIVE_RESULTS / f'seed_{SEED}'
DEST.mkdir(parents=True, exist_ok=True)
shutil.copytree(LOCAL_RESULT, DEST, dirs_exist_ok=True)

# Only a compact summary is parsed; the actual evaluator and saved weights are the authors'.
text = log_path.read_text()
values = re.findall(r'Test Evaluation: acc: ([0-9.]+), kappa: ([0-9.]+), f1: ([0-9.]+)', text)
if not values:
    raise RuntimeError("Could not find the authors' test metrics in train.log")
acc, kappa, f1 = map(float, values[-1])
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
with (DRIVE_RESULTS / 'test_scores.csv').open('w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['seed', 'balanced_accuracy', 'kappa', 'weighted_f1'])
    writer.writeheader()
    writer.writerow(dict(seed=SEED, balanced_accuracy=acc, kappa=kappa, weighted_f1=f1))
(DRIVE_RESULTS / 'authors_commit.txt').write_text(OFFICIAL_COMMIT + '\n')
print(f'Test balanced accuracy: {acc:.5f}; kappa: {kappa:.5f}; weighted F1: {f1:.5f}')
print('Saved final model, training log, and scores:', DRIVE_RESULTS)

Test balanced accuracy: 0.63457; kappa: 0.51269; weighted F1: 0.63527
Saved final model, training log, and scores: /content/drive/MyDrive/CBraMod-MI/results/physionet_mi_4class


## Folder organization

```text
MyDrive/CBraMod-MI/
├── eeg-motor-movementimagery-dataset-1.0.0.zip  # original, kept for your 00 setup notebook
├── data/physionet_mi/
│   ├── files/S001/S001R04.edf ...               # extracted once, kept on Drive
│   └── processed_average/data.mdb               # authors' LMDB, reused next session
├── models/pretrained/cbramod/pretrained_weights.pth
└── results/physionet_mi_4class/
    ├── authors_commit.txt
    ├── test_scores.csv
    └── seed_3407/
        ├── train.log
        └── epoch..._acc_..._kappa_..._f1_....pth

/content/                                     # temporary; recreated each Colab session
├── cbramod-head-interp/                       # your existing setup notebook clones this
├── CBraMod/                                   # official authors' source
└── data/physionet_mi/                        # local copy used for preprocessing/training
```

**What stays scientifically unchanged:** the authors' EDF preprocessing, subject split, trial format/LMDB, pretrained CBraMod architecture and weights, training settings, model selection, and evaluation. The new notebook only manages Colab storage and a small LMDB loader connection compatibility issue. It has not been run end-to-end on a Colab A100 here.